# METAR Sanity Check — NZWN 2009–2026Coverage, missingness, temperature distribution by CP hour.Purely exploratory — not load-bearing.

In [ ]:
import polars as plimport datetime as dtfrom solarstorm.data._iem import fetch_iem_asosfrom solarstorm.data._metar import parse_tmp_c_int_from_rowfrom pathlib import Pathcache = Path('./.cache/iem')df = fetch_iem_asos('NZWN', dt.date(2009,1,1), dt.date(2026,6,3), cache_dir=cache)print(f'{df.height:,} rows, {df["valid"].n_unique():,} unique timestamps')

In [ ]:
# Parse all rowsstats = {"n_total":0,"n_ok":0,"n_imputed":0,"n_missing":0,"n_implausible":0}tts = []for row in df.iter_rows(named=True):    tt, _, dq, implausible = parse_tmp_c_int_from_row(row['metar'], row.get('tmpf'))    stats['n_total'] += 1    stats[f'n_{dq}'] += 1    if implausible: stats['n_implausible'] += 1    if tt is not None: tts.append(tt)print(stats)

In [ ]:
# Coverage by local hourdf_hour = df.with_columns(pl.col('valid').dt.convert_time_zone('Pacific/Auckland').dt.hour().alias('hour_local'))hour_counts = df_hour.group_by('hour_local').count().sort('hour_local')hour_counts